In [0]:
%sql
SELECT * FROM cleaned_taxi_data

### Question 1

In [0]:
%sql
WITH monthly_stats AS (
    SELECT
        DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS year_month,
        ROUND(avg(passenger_count)) AS avg_passenger_count,
        ROUND(avg(total_amount), 2) AS avg_amount_paid,
        ROUND(avg(total_amount / passenger_count), 2) AS avg_amount_per_passenger,
        COUNT(*) AS total_trips
    FROM cleaned_taxi_data
    GROUP BY 1
),
monthly_day_stats AS (
    SELECT
        DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS year_month,
        DATE_FORMAT(pickup_datetime, 'EEEE') AS day_of_week,
        HOUR(pickup_datetime) AS hour_of_day,
        COUNT(*) AS trips
    FROM cleaned_taxi_data
    GROUP BY 1, 2, 3
),
ranked_days AS (
    SELECT
        mds.year_month,
        mds.day_of_week,
        mds.hour_of_day,
        mds.trips,
        ROW_NUMBER() OVER (PARTITION BY mds.year_month ORDER BY mds.trips DESC) AS rn
    FROM monthly_day_stats mds
)
SELECT
    ms.year_month,
    ms.total_trips,
    ms.avg_passenger_count,
    ms.avg_amount_paid,
    ms.avg_amount_per_passenger,
    rd.day_of_week AS busiest_day,
    rd.hour_of_day AS busiest_hour
FROM monthly_stats ms
JOIN ranked_days rd
  ON ms.year_month = rd.year_month
WHERE rd.rn = 1
ORDER BY ms.year_month;


## Question 2

In [0]:
%sql
WITH taxi_metrics AS (
    SELECT
        taxi_type,
        UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime) AS trip_duration_sec,
        trip_distance * 1.60934 AS trip_distance_km,
        speed_mph * 1.60934 AS speed_km
    FROM cleaned_taxi_data
)
SELECT
    taxi_type,
    ROUND(AVG(trip_duration_sec) / 60, 2) AS avg_duration_min,
    ROUND(PERCENTILE_APPROX(trip_duration_sec, 0.5) / 60, 2) AS median_duration_min,
    ROUND(MIN(trip_duration_sec) / 60, 2) AS min_duration_min,
    ROUND(MAX(trip_duration_sec) / 60, 2) AS max_duration_min,
    ROUND(AVG(trip_distance_km), 2) AS avg_trip_distance_km,
    ROUND(PERCENTILE_APPROX(trip_distance_km, 0.5), 2) AS median_trip_distance_km,
    ROUND(MIN(trip_distance_km), 2) AS min_trip_distance_km,
    ROUND(MAX(trip_distance_km), 2) AS max_trip_distance_km,
    ROUND(AVG(speed_km), 2) AS avg_speed_km,
    ROUND(PERCENTILE_APPROX(speed_km, 0.5), 2) AS median_speed_km,
    ROUND(MIN(speed_km), 2) AS min_speed_km,
    ROUND(MAX(speed_km), 2) AS max_speed_km
FROM taxi_metrics
GROUP BY taxi_type;


## Question 3

In [0]:
%sql
SELECT
    taxi_type,
    pickup_borough,
    dropoff_borough,
    DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS year_month,  
    DATE_FORMAT(pickup_datetime, 'MMM') AS month,
    DATE_FORMAT(pickup_datetime, 'EEEE') AS day_of_week,
    HOUR(pickup_datetime) AS hour_of_day,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance * 1.60934), 2) AS avg_distance_km,
    ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,
    ROUND(SUM(total_amount), 2) AS total_amount_paid
 

FROM cleaned_taxi_data
GROUP BY
    taxi_type,
    pickup_borough,
    dropoff_borough,
    year_month,  
    month,
    day_of_week,
    hour_of_day;


## Question 4

In [0]:
%sql
WITH pair_revenue AS (
    SELECT
        pickup_borough,
        dropoff_borough,
        SUM(total_amount) AS total_revenue
    FROM cleaned_taxi_data
    WHERE YEAR(pickup_datetime) = 2024
    GROUP BY pickup_borough, dropoff_borough
),
ranked_pairs AS (
    SELECT
        pickup_borough,
        dropoff_borough,
        total_revenue,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
    FROM pair_revenue
),
total_revenue_2024 AS (
    SELECT SUM(total_revenue) AS total_revenue_2024
    FROM pair_revenue
)
SELECT
    rp.pickup_borough,
    rp.dropoff_borough,
    rp.total_revenue,
    ROUND(rp.total_revenue / tr.total_revenue_2024 * 100, 2) AS revenue_share_percent
FROM ranked_pairs rp
CROSS JOIN total_revenue_2024 tr
WHERE rp.revenue_rank <= 10
ORDER BY rp.revenue_rank;


## Question 5

In [0]:
%sql
SELECT 
    ROUND(SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS tip_percentage
FROM cleaned_taxi_data
    

## Question 6

In [0]:
%sql
SELECT 
    ROUND(SUM(CASE WHEN tip_amount >= 15 THEN 1 ELSE 0 END) / SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) * 100, 2) AS 15_plus_tip_percentage 
FROM cleaned_taxi_data

## Question 7


In [0]:
%sql
SELECT
    CASE
        WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 < 5 THEN 'Under 5 mins'
        WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 5 AND 10 THEN '5-10 mins'
        WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 10 AND 20 THEN '10-20 mins'
        WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 20 AND 30 THEN '20-30 mins'
        WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 30 AND 60 THEN '30-60 mins'
        ELSE '60+ min'
    END AS trip_duration_category,
  
    ROUND(AVG(speed_mph * 1.60934), 2) AS avg_speed_kmh,
    ROUND(AVG((trip_distance * 1.60934) / total_amount), 2) AS avg_distance_per_dollar

FROM cleaned_taxi_data
WHERE total_amount > 0
    GROUP BY
        CASE
          WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 < 5 THEN 'Under 5 mins'
          WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 5 AND 10 THEN '5-10 mins'
          WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 10 AND 20 THEN '10-20 mins'
          WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 20 AND 30 THEN '20-30 mins'
          WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 30 AND 60 THEN '30-60 mins'
          ELSE '60+ min'
        END 





## Question 8

In [0]:
%sql
WITH trips_with_category AS (
    SELECT
        *,
        (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 AS trip_duration_min,
        CASE
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 < 5 THEN 'Under 5 mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 5 AND 10 THEN '5-10 mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 10 AND 20 THEN '10-20 mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 20 AND 30 THEN '20-30 mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 BETWEEN 30 AND 60 THEN '30-60 mins'
            ELSE '60+ mins'
        END AS trip_duration_category
    FROM cleaned_taxi_data
    WHERE total_amount > 0
)
SELECT
    trip_duration_category,
    COUNT(*) AS num_trips,
    ROUND(AVG(total_amount), 2) AS avg_fare_per_trip,
    ROUND(AVG(trip_duration_min), 2) AS avg_duration_min,
    ROUND(AVG(total_amount) / AVG(trip_duration_min) * 60, 2) AS estimated_income_per_hour
FROM trips_with_category
GROUP BY trip_duration_category
ORDER BY estimated_income_per_hour DESC;
